In [ ]:
import sys
import os
import time
sys.dont_write_bytecode = True

sys.path.insert(0, "/model/src/")
from scvi.model._totalvi import TOTALVI

import torch
import numpy as np
import mudata as md
import scanpy as sc
import pandas as pd

from graph.graph import prep_graph_splits


# Paths

base_dir = os.path.dirname(os.path.abspath(__file__))
path_to_mdata = os.path.join(base_dir, "../data/tonsil/tonsil_pp_svg.h5mu")
path_to_full_graph = os.path.join(base_dir, "../data/graph/graph_k5_sim_inv.pt")
path_to_split_graph = os.path.join(base_dir, "../data/graph/k5_sim_inv_train_val_subgraphs.pt")
path_to_model = os.path.join(base_dir, "../data/trained_models/sim_inv/")


#load data 
mdata = md.read_h5mu(path_to_mdata)
n_cells = mdata["RNA"].shape[0]
print(f"Total cells = {n_cells}")

# percentages to test
subset_fracs = [0.2, 0.4, 0.6, 0.8, 1.0]
results = []

# Loop through dataset sizes
for frac in subset_fracs:
    print("\n==============================")
    print(f" Running subset: {int(frac*100)}% ")
    print("==============================")

    #subset dataset 
    n_sub = int(n_cells * frac)
    cell_idx = np.random.choice(n_cells, n_sub, replace=False)

    m_sub = mdata[cell_idx].copy()

    #subset graphs
    subgraphs = prep_graph_splits(
        m_sub,
        path_to_graph=path_to_full_graph,
        path_to_save_graph_splits=path_to_split_graph,
        train_split=0.75
    )

    #setup TOTALVI
    TOTALVI.setup_mudata(
        m_sub,
        rna_layer=None,
        protein_layer=None,
        modalities={"rna_layer": "RNA", "protein_layer": "Protein"}
    )

    model = TOTALVI(
        m_sub,
        path_to_graphs=path_to_split_graph,
        graph_n_layers=1,
        graph_conv_type="SGC",
        graph_norm_type="layer",
        graph_act_type="elu",
        graph_sgc_Kparam=1
    )

    # ---- Training ----
    start = time.time()

    model.train(
        max_epochs=400,
        batch_size=len(subgraphs["train_indices"]),
        external_indexing=[subgraphs["train_indices"], subgraphs["val_indices"]]
    )

    end = time.time()
    runtime = end - start

    print(f"Runtime for {int(frac*100)}% subset: {runtime:.2f} sec")

    # save model individually
    model_path = os.path.join(path_to_model, f"subset_{int(frac*100)}.pt")
    model.save(model_path)

    results.append({
        "subset_percent": frac * 100,
        "n_cells": n_sub,
        "runtime_sec": runtime
    })


# Save results to CSV

df = pd.DataFrame(results)
df.to_csv("runtime_results.csv", index=False)

print("\nSaved runtime_results.csv")


## Plotting

In [ ]:
library(ggplot2)
library(readr)

df <- read_csv("runtime_results.csv")

ggplot(df, aes(x = subset_percent, y = runtime_sec)) +
  geom_line(size = 1.2) +
  geom_point(size = 3) +
  theme_minimal(base_size = 14) +
  labs(
    title = "TOTALVI Runtime vs Dataset Size",
    x = "Dataset size (%)",
    y = "Runtime (seconds)"
  )

